In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col


In [0]:
RENAME_MAP = {
    "cid": "company_id",
    "cntry": "country"
}

Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101_raw")

#### 1. Triming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

#### 2. Derivation and Normalization

In [0]:
df = (
    df
    # Remove dashes from company id
    .withColumn(
        "cid",
        F.regexp_replace(col("cid"), "-", "")
    )

    # Normalize country codes
    .withColumn(
        "cntry",
        F.when(col("cntry") == "DE", "Germany")
         .when(col("cntry").isin("US", "USA"), "United States")
         .when((col("cntry") == "") | col("cntry").isNull(), "n/a")
         .otherwise(col("cntry"))
    )
)

#### 3. Renaming

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
table_name = "silver.erp_customer_location"
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

In [0]:
(
    df.write
      .mode("overwrite")
      .format("delta")
      .saveAsTable("silver.erp_customer_location")
)

In [0]:
%sql
select * from silver.erp_customer_location limit 100